# MLFlow Manual Logging into File

This notebook demonstrates manual logging of machine learning experiments using MLflow with scikit-learn. It covers the following steps:

- Loading and preprocessing the Statlog (German Credit Data) dataset.
- Training a RandomForestClassifier and logging parameters, metrics, and artifacts to MLflow.
- Saving and loading models using MLflow's model registry.
- Performing hyperparameter optimization with Hyperopt and logging the best model.

The notebook provides practical examples for experiment tracking, model management, and reproducibility in ML workflows.

In [1]:
!pip install ucimlrepo
!pip install "azureml-mlflow==1.59.0"
!pip install hyperopt

In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Fetch Statlog (German Credit Data) dataset
credit_data = fetch_ucirepo(id=27)

# Features and targets
X = credit_data.data.features
y = credit_data.data.targets

# Categorical encoding for X and y
X = pd.get_dummies(X)
le = LabelEncoder()
y = le.fit_transform(y)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

# Define the model
model = RandomForestClassifier(random_state=42)

/anaconda/envs/azureml_py38/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


## 1. Basic Logging

In [3]:
# Create confusion matrix plot and save as artifact

def confusion_matrix_artifact(y_true, y_pred):
    import matplotlib.pyplot as plt
    from sklearn.metrics import confusion_matrix
    import seaborn as sns

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots()
    sns.heatmap(cm, annot=True, fmt='d', ax=ax)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.close(fig)
    return fig

### MLflow Logging: Training and Tracking a RandomForest Model

This cell demonstrates how to manually log a scikit-learn RandomForestClassifier training run using MLflow. The following steps are performed:

1. **MLflow Setup**
   - `mlflow.set_tracking_uri("file:../mlruns")`: Specifies where MLflow will store all experiment runs and artifacts. Using a local file path makes it easy to inspect results and share with others.
   - `mlflow.set_experiment("SklearnExperiment")`: Creates or selects an experiment named 'SklearnExperiment'. Experiments are containers for related runs, helping organize and compare results.

2. **Start a Run**
   - `with mlflow.start_run()`: Opens a context for a new MLflow run. All logging (parameters, metrics, artifacts, models) within this block is grouped under a single run, making it easy to track and reproduce.

3. **Parameter Logging**
   - `mlflow.log_param("n_estimators", n_estimators)` and `mlflow.log_param("max_depth", max_depth)`: Records the hyperparameters used for this model. This is crucial for experiment tracking and later analysis, allowing you to compare how different parameter choices affect performance.

4. **Model Training**
   - The RandomForestClassifier is instantiated and trained on the training data. This step produces the fitted model used for evaluation and logging.

5. **Metric Logging**
   - `mlflow.log_metric("score", score)`: Logs the model's accuracy on the test set. Metrics are tracked over time and across runs, enabling performance comparisons and trend analysis.

6. **Artifact Logging**
   - `mlflow.log_figure(cm_figure, "confusion_matrix.png")`: Saves the confusion matrix plot as an artifact. Artifacts can include plots, data files, or any output relevant to the run. They are stored with the run and accessible via the MLflow UI.

7. **Tagging**
   - `mlflow.set_tag("Training Info", "Basic model for German Credit Data")`: Adds a custom tag to the run. Tags are useful for annotating runs with additional context, such as dataset name, model type, or experiment purpose.

8. **Model Logging**
   - `mlflow.sklearn.log_model(rf, "credit_rf_model", signature=signature)`: Logs the trained model to MLflow, including its input/output signature. This enables model versioning, reproducibility, and easy deployment. The signature documents the expected input and output formats.

9. **Model URI Output**
   - The URI of the logged model is printed. This URI can be used to load the model for inference or further analysis in future workflows.

By following these steps, you ensure that every aspect of your machine learning experiment is tracked, reproducible, and easy to analyze or share. MLflow's UI and APIs make it simple to review past runs, compare models, and deploy the best-performing versions.


In [5]:
import mlflow
from mlflow.models import infer_signature

mlflow.set_tracking_uri("file:./mlflow_experiments")
mlflow.set_experiment("SklearnExperiment")

with mlflow.start_run():
    n_estimators = 100
    max_depth = 6
    print(f"Logging parameters: n_estimators={n_estimators}, max_depth={max_depth}")
    # TODO Log parameters
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)

    print("Training RandomForestClassifier...")
    rf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth)
    rf.fit(X_train, y_train)

    score = rf.score(X_test, y_test)
    print(f"Test accuracy (score): {score:.4f}")
    mlflow.log_metric("score", score)

    print("Generating confusion matrix plot and logging as artifact...")
    y_pred = rf.predict(X_test)
    cm_figure = confusion_matrix_artifact(y_test, y_pred)
    mlflow.log_figure(cm_figure, "confusion_matrix.png")

    print("Setting MLflow tag for run description...")
    mlflow.set_tag("Training Info", "Basic model for German Credit Data")

    print("Logging model to MLflow...")
    signature = infer_signature(X_train, rf.predict(X_train))
    model_info = mlflow.sklearn.log_model(
        rf, "credit_rf_model", signature=signature)
    print("Model logged at URI:", model_info.model_uri)

Logging parameters: n_estimators=100, max_depth=6
Training RandomForestClassifier...
Test accuracy (score): 0.8696
Generating confusion matrix plot and logging as artifact...
Model logged at URI: runs:/3ccdda6a307842f29e6fcab5cb0902df/credit_rf_model


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/mlflow/types/utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/02/19 10:16:21 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - mlflow (current: 2.19.0, required: mlflow==2.15.1)
To fix the mism

In [6]:
model_info.model_uri

'runs:/3ccdda6a307842f29e6fcab5cb0902df/credit_rf_model'

In [7]:
# Load the trained model from MLflow using its URI.
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)

# Use the loaded model to make predictions on the test set.
y_pred = loaded_model.predict(X_test)

# Import accuracy_score to evaluate the predictions.
from sklearn.metrics import accuracy_score

# Calculate the accuracy of the reloaded model on the test set.
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy to verify the loaded model's performance.
print(f"Test Accuracy: {test_accuracy:.4f}")

2026/02/19 10:19:17 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - mlflow (current: 2.19.0, required: mlflow==2.15.1)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


Test Accuracy: 0.8696


## 2. Hyperparameter Tuning with hyperopt

In [8]:
import mlflow
from mlflow.models import infer_signature
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np

mlflow.set_tracking_uri("file:./mlflow_experiments")

# Create a new MLflow Experiment
mlflow.set_experiment("Hyperopt_SklearnExperiment")

# Define the objective function for hyperparameter optimization
def objective(params):
    with mlflow.start_run(nested=True):
        # Log hyperparameters
        mlflow.log_params(params)

        # Create and train the model with given parameters
        rf = RandomForestClassifier(
            n_estimators=int(params['n_estimators']),
            max_depth=int(params['max_depth']),
            random_state=42
        )
        rf.fit(X_train, y_train)

        # Predict and evaluate
        y_pred = rf.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        
        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        
        # Log the model with MLflow
        # signature = infer_signature(X_train, rf.predict(X_train))
        # mlflow.sklearn.log_model(rf, "rf_model", signature=signature)
        
        # Return the metric for optimization
        return {'loss': -accuracy, 'status': STATUS_OK}

# Define the search space
search_space = {
    'n_estimators': hp.quniform('n_estimators', 50, 200, 10),
    'max_depth': hp.quniform('max_depth', 4, 20, 1)
}

# Run the hyperparameter optimization
trials = Trials()
best_params = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=10,
    trials=trials
)

# Log the best parameters found
with mlflow.start_run(run_name="RandomForest_Hyperopt"):
    mlflow.log_params(best_params)

    # Train and log the best model
    best_rf = RandomForestClassifier(
        n_estimators=int(best_params['n_estimators']),
        max_depth=int(best_params['max_depth']),
        random_state=42
    )
    best_rf.fit(X_train, y_train)

    best_score = best_rf.score(X_test, y_test)
    mlflow.log_metric("best_accuracy", best_score)

    signature = infer_signature(X_train, best_rf.predict(X_train))
    model_info = mlflow.sklearn.log_model(best_rf, "best_rf_model", signature=signature)
    
    print("Best model trained and logged with accuracy:", best_score)

2026/02/19 10:23:38 INFO mlflow.tracking.fluent: Experiment with name 'Hyperopt_SklearnExperiment' does not exist. Creating a new experiment.
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/mlflow/types/utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/02/19 10:24:12 WARNING mlflow.utils.requirements_utils: Detected one or more mismatc

100%|██████████| 10/10 [00:26<00:00,  2.63s/trial, best loss: -0.8731884057971014]
Best model trained and logged with accuracy: 0.8731884057971014


#### Detailed MLflow Logging Steps

This code block demonstrates how to use MLflow for experiment tracking and model management with scikit-learn. Here is a breakdown of each operation:

1. **MLflow Setup**
   - `mlflow.set_tracking_uri("file:../mlruns")`: Sets the location for storing MLflow runs and artifacts. Here, it uses a local directory.
   - `mlflow.set_experiment("SklearnExperiment")`: Creates or selects an experiment named 'SklearnExperiment'. All runs will be grouped under this experiment.

2. **Start a Run**
   - `with mlflow.start_run()`: Begins a new MLflow run context. All logging within this block is associated with this run.

3. **Parameter Logging**
   - `mlflow.log_param(...)`: Records hyperparameters (`n_estimators`, `max_depth`) for the model. These are useful for comparing runs.

4. **Model Training**
   - Trains a `RandomForestClassifier` using the specified parameters and training data.

5. **Metric Logging**
   - `mlflow.log_metric("score", score)`: Logs the test accuracy of the model. Metrics help track model performance over different runs.

6. **Artifact Logging**
   - `mlflow.log_figure(cm_figure, "confusion_matrix.png")`: Saves the confusion matrix plot as an artifact. Artifacts can be any files (plots, models, etc.) associated with the run.

7. **Tagging**
   - `mlflow.set_tag(...)`: Adds a descriptive tag to the run, making it easier to search and organize runs in the MLflow UI.

8. **Model Logging**
   - `mlflow.sklearn.log_model(...)`: Saves the trained model to MLflow, including its input/output signature for reproducibility. The model can later be loaded and used for inference.

9. **Model URI Output**
   - Prints the URI where the model is stored. This URI can be used to load the model in future workflows.

Each step ensures that the experiment is fully tracked, reproducible, and easy to compare with other runs using MLflow's UI and APIs.


#### Reloading and Using a Model from MLflow

After logging a trained model to MLflow, you can reload it for inference or further evaluation. This process ensures reproducibility and enables model deployment in different environments.

- `mlflow.pyfunc.load_model(model_info.model_uri)`: Loads the model from the MLflow registry using its URI. The loaded model can be used for predictions just like the original scikit-learn model.